1. Imports and files

In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/genai-hybrid-skills-adaptation/output/Occupational"
)

VALIDATION_FILE = (
    BASE_DIR / "Validation" / "Occupational_Task_Validation_Sample_v6.csv"
)

ELOUNDOU_URL = (
    "https://raw.githubusercontent.com/openai/"
    "GPTs-are-GPTs/refs/heads/main/data/full_labelset.tsv"
)

print("Validation file:", VALIDATION_FILE)
print("External source:", ELOUNDOU_URL)

2. Load our frozen sample

In [ ]:
our_tasks = pd.read_csv(VALIDATION_FILE)

print("Tasks loaded:", len(our_tasks))
print("Unique task IDs:", our_tasks["task_id"].nunique())
print(
    "Missing provisional labels:",
    our_tasks["provisional_label"].isna().sum()
)

3. Load published task annotations

In [ ]:
eloundou = pd.read_csv(
    ELOUNDOU_URL,
    sep="\t",
    index_col=0
)

eloundou["Task ID"] = eloundou["Task ID"].astype("Int64")

print("Published tasks loaded:", len(eloundou))
print("Unique published Task IDs:", eloundou["Task ID"].nunique())

4. Prepare only the published fields we actually use

In [ ]:
eloundou_aligned = (
    eloundou[
        [
            "Task ID",
            "O*NET-SOC Code",
            "Title",
            "Task",
            "human_exposure_agg",
            "gpt4_automation"
        ]
    ]
    .rename(
        columns={
            "Task ID": "task_id",
            "O*NET-SOC Code": "onet_soc_code_published",
            "Title": "occupation_title_published",
            "Task": "task_statement_published"
        }
    )
)

eloundou_aligned["soc6_published"] = (
    eloundou_aligned["onet_soc_code_published"]
    .astype(str)
    .str.extract(r"(\d{2}-\d{4})")
)

print("Published comparison fields prepared.")

5. Exact Task-ID match

In [ ]:
external_compare = our_tasks[
    [
        "task_id",
        "soc6",
        "occupation_title",
        "task_statement",
        "provisional_label",
        "sampling_weight"
    ]
].merge(
    eloundou_aligned,
    on="task_id",
    how="left"
)

external_compare["soc_match"] = (
    external_compare["soc6"]
    == external_compare["soc6_published"]
)

external_compare["text_match"] = (
    external_compare["task_statement"]
    == external_compare["task_statement_published"]
)

print("Our tasks:", len(our_tasks))
print(
    "Exact Task-ID matches:",
    external_compare["human_exposure_agg"].notna().sum()
)
print(
    "SOC matches:",
    f"{external_compare['soc_match'].sum()}/{len(external_compare)}"
)
print(
    "Exact task-text matches:",
    f"{external_compare['text_match'].sum()}/{len(external_compare)}"
)

6. Directional convergence rules

In [ ]:
def exposure_aligned(row):
    label = row["provisional_label"]

    if label == "Limited GenAI applicability":
        return row["human_exposure_agg"] == "E0"

    if label in [
        "Augmentation-oriented",
        "Automation-oriented"
    ]:
        return row["human_exposure_agg"] in ["E1", "E2"]

    return pd.NA


def automation_aligned(row):
    label = row["provisional_label"]

    if label == "Limited GenAI applicability":
        return row["gpt4_automation"] in ["T0", "T1"]

    if label == "Augmentation-oriented":
        return row["gpt4_automation"] in ["T2", "T3"]

    if label == "Automation-oriented":
        return row["gpt4_automation"] in ["T3", "T4"]

    return pd.NA


external_compare["exposure_aligned"] = external_compare.apply(
    exposure_aligned,
    axis=1
)

external_compare["automation_aligned"] = external_compare.apply(
    automation_aligned,
    axis=1
)

scored = external_compare[
    external_compare["provisional_label"]
    != "Indeterminate mechanism"
].copy()

print("Tasks scored for directional convergence:", len(scored))
print(
    "Indeterminate tasks excluded:",
    (external_compare["provisional_label"]
     == "Indeterminate mechanism").sum()
)

7. Print the only results worth retaining

In [ ]:
print(
    "Human-exposure directional alignment:",
    f"{scored['exposure_aligned'].mean():.1%}"
)

print(
    "GPT-4 automation directional alignment:",
    f"{scored['automation_aligned'].mean():.1%}"
)

for label in [
    "Limited GenAI applicability",
    "Augmentation-oriented",
    "Automation-oriented"
]:
    subset = scored[
        scored["provisional_label"] == label
    ]

    print(f"\n{label}")
    print(
        "  Human exposure:",
        f"{subset['exposure_aligned'].mean():.1%}"
    )
    print(
        "  GPT-4 automation:",
        f"{subset['automation_aligned'].mean():.1%}"
    )

8. Save the audit file

In [ ]:
OUTPUT_FILE = (
    BASE_DIR
    / "Validation"
    / "Occupational_Task_External_Convergence_v6.csv"
)

external_compare.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved:", OUTPUT_FILE)
print("Rows:", len(external_compare))